In [1]:
import pandas as pd
import json
from pathlib import Path

# Load the 250 annotation drafts
path = Path("../data/week3_annotation_draft.jsonl")

with path.open("r", encoding="utf-8") as f:
    annotations = [json.loads(line) for line in f]

df = pd.DataFrame(annotations)

print("Total remarks:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

Total remarks: 50
Columns: ['id', 'text', 'entities', 'verified']


,id,text,entities,verified
0,142,"Picture your mornings here: coffee in hand, lo...","[{'start': 579, 'end': 594, 'label': 'BATHROOM...",True
1,6,Welcome to this beautifully maintained 2021-bu...,"[{'start': 114, 'end': 124, 'label': 'BEDROOMS...",True
2,97,"Step into timeless charm at 19 Balsam Drive, a...","[{'start': 309, 'end': 319, 'label': 'BEDROOMS...",True
3,60,"PRICE IMPROVED! Welcome to 26249 Fleet Ln, Hel...","[{'start': 71, 'end': 81, 'label': 'BEDROOMS'}...",True
4,112,NOW AVAILABLE - PRICED TO SELL IN THE FAIRWAYS...,"[{'start': 284, 'end': 300, 'label': 'SQFT'}, ...",True


In [2]:
import sys
from pathlib import Path

# Add the main project directory to Python's search path
sys.path.insert(0, str(Path.cwd().parent))

from scripts.entity_extractor import EntityExtractor

extractor = EntityExtractor()

examples = [
    "three bathrooms",
    "3 full bathrooms",
    "2.5 baths",
    "3ba",
    "3-bedroom, 2-bathroom",
    "four bedrooms and two bathrooms",
]

for text in examples:
    print(text)
    print(extractor.extract_all(text))
    print()

three bathrooms
{'bedrooms': None, 'bathrooms': 3, 'price': None, 'sqft': None, 'amenities': []}

3 full bathrooms
{'bedrooms': None, 'bathrooms': 3, 'price': None, 'sqft': None, 'amenities': []}

2.5 baths
{'bedrooms': None, 'bathrooms': 2.5, 'price': None, 'sqft': None, 'amenities': []}

3ba
{'bedrooms': None, 'bathrooms': 3, 'price': None, 'sqft': None, 'amenities': []}

3-bedroom, 2-bathroom
{'bedrooms': 3, 'bathrooms': 2, 'price': None, 'sqft': None, 'amenities': []}

four bedrooms and two bathrooms
{'bedrooms': 4, 'bathrooms': 2, 'price': None, 'sqft': None, 'amenities': []}



In [3]:
import json

with open("../data/week3_annotation_draft.jsonl", encoding="utf-8") as f:
    annotations = [json.loads(line) for line in f]

record = next(r for r in annotations if r["id"] == 142)

print("Listing ID:", record["id"])

for entity in record["entities"]:
    print(
        entity["label"],
        "->",
        record["text"][entity["start"]:entity["end"]]
    )

Listing ID: 142
BATHROOMS -> three bathrooms


In [4]:
import json

with open("../data/week3_gold_test.jsonl", encoding="utf-8") as f:
    gold_records = [json.loads(line) for line in f]

verified = sum(r["verified"] for r in gold_records)

print(f"Verified: {verified}/{len(gold_records)}")

Verified: 1/50


In [5]:

def review_next():
    """Display the next unverified listing."""
    for i, record in enumerate(gold_records):
        if not record["verified"]:
            print(f"\nRecord Index: {i}")
            print(f"Listing ID: {record['id']}")
            print("\nTEXT:\n", record["text"])

            print("\nENTITIES:")
            for j, entity in enumerate(record["entities"]):
                value = record["text"][
                    entity["start"]:entity["end"]
                ]
                print(f"{j}: {entity['label']} -> {value}")

            return i

    print("All listings are verified!")
    return None


def add_entity(index, phrase, label, occurrence=0):
    """Add a missing entity by its exact text."""
    record = gold_records[index]
    text = record["text"]

    matches = [
        i for i in range(len(text))
        if text.startswith(phrase, i)
    ]

    if occurrence >= len(matches):
        print("Phrase not found.")
        return

    start = matches[occurrence]

    entity = {
        "start": start,
        "end": start + len(phrase),
        "label": label.upper()
    }

    if entity not in record["entities"]:
        record["entities"].append(entity)

    print(f"Added {label.upper()}: {phrase}")


def remove_entity(index, entity_index):
    """Remove an incorrectly labeled entity."""
    removed = gold_records[index]["entities"].pop(entity_index)
    print("Removed:", removed)


def save_gold():
    """Save corrected annotations to the JSONL file."""
    with open(path, "w", encoding="utf-8") as f:
        for record in gold_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(
        "Verified:",
        sum(r["verified"] for r in gold_records),
        "/",
        len(gold_records)
    )


In [6]:
current_index = review_next()


Record Index: 0
Listing ID: 142

TEXT:
 Picture your mornings here: coffee in hand, looking out past the treetops toward the mountains, with nothing on the schedule but deciding how to spend the day. That's the rhythm this Lake Arrowhead home-with lake rights-sets. Upstairs, the living space centers around those views-treetops and mountain peaks framing the room like a living painting. It's the kind of spot that works just as well for a quiet Sunday as it does for a house full of guests. The kitchen sits right where you want it, and the main-level suite means no stairs when you're ready to call it a night. With three bathrooms spread throughout, there's never a lineup, even when the place is full. But the real surprise is downstairs. The oversized game room practically begs for a purpose-is it game night? A movie room? A second hangout space where kids and adults can split off? After a day on the water, it's the kind of room where everyone naturally ends up, shoes off, laughing about 

In [7]:
add_entity(current_index, "three bathrooms", "BATHROOMS")

Added BATHROOMS: three bathrooms


In [8]:
record = gold_records[current_index]

for entity in record["entities"]:
    print(
        entity["label"],
        "->",
        record["text"][entity["start"]:entity["end"]]
    )

BATHROOMS -> three bathrooms


In [9]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 2 / 50


In [10]:
current_index = review_next()



Record Index: 2
Listing ID: 97

TEXT:
 Step into timeless charm at 19 Balsam Drive, a delightful Spanish-style bungalow nestled in the heart of National City. Built in 1930, this character-filled residence showcases the warmth and architectural appeal that make Spanish homes so beloved, blending vintage charm with everyday comfort. This home has 2 bedrooms, 1 bathroom, and 859 square feet of living space, the home features an inviting layout filled with natural light and classic details that reflect its historic character. Situated on a 2,682-square-foot lot, the property has a private outdoor space perfect for relaxing, gardening, or enjoying Southern California's year-round sunshine. Ideally located just minutes from Naval Base San Diego, this home presents an excellent opportunity for military personnel, first-time buyers, or investors seeking a property in a convenient and desirable location. Enjoy easy access to shopping, dining, schools, parks, and major transportation routes, a

In [11]:
gold_records[current_index]["verified"] = True
save_gold()


Verified: 3 / 50


In [12]:
current_index = review_next()


Record Index: 3
Listing ID: 60

TEXT:
 PRICE IMPROVED! Welcome to 26249 Fleet Ln, Helendale. This home offers 4 bedrooms plus a private office, 2 baths, and 2,082 square feet of thoughtfully updated living space on a 7,919 square foot lot in the desirable Silver Lakes community, with your own private dock right out your back door! Step inside to a freshly painted interior and brand new laminate wood-look flooring that flows throughout the home. Recessed lighting has been added in the main living areas for a clean, modern look, and ceiling fans have been installed throughout to keep every room comfortable. The primary suite is a true highlight with direct access to your backyard and lake beyond. The bathroom has been completely updated with an all-glass walk-in shower featuring a built-in bench, alongside a deep soaking tub for a spa-like retreat. Outside, the front and side yards have been transformed with low-maintenance artificial turf, giving the home great curb appeal without the 

In [13]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 4 / 50


In [14]:
current_index = review_next()


Record Index: 4
Listing ID: 112

TEXT:
 NOW AVAILABLE - PRICED TO SELL IN THE FAIRWAYS AT REDHAWK! Welcome to this beautifully upgraded home located in the highly sought-after gated community of The Fairways at Redhawk in South Temecula. Situated on a desirable corner lot, this spacious two-story home offers approximately 2478 square feet of living space with 4 bedrooms and 2.5 bathrooms, combining comfort, style, and functionality in an exceptional South Temecula location. Step inside to a bright and inviting floor plan featuring granite kitchen countertops, tile and laminate wood flooring, and generous living spaces designed for everyday living and entertaining. The spacious kitchen features a large center island and opens directly to the family room with a cozy fireplace, creating a natural gathering place for family and friends. The main level includes a formal living room, spacious family room, open kitchen, and convenient guest half-bathroom. Upstairs are all four bedrooms, two 

In [15]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 5 / 50


In [16]:
current_index = review_next()


Record Index: 5
Listing ID: 181

TEXT:
 Welcome to this beautifully renovated 3-bedroom, 2-bathroom home featuring a spacious bonus room and an impressive list of upgrades. The home boasts a brand-new roof and a gorgeous, fully updated kitchen with modern finishes and plenty of space for entertaining. Situated on a huge corner lot, this property offers excellent outdoor space, convenient RV access, and incredible potential for future expansion. The generous lot may provide an opportunity to add an ADU (Accessory Dwelling Unit), buyer to verify with the city and applicable zoning requirements. With its functional floor plan, stylish renovations, oversized lot, RV access, and exciting potential for additional living space, this property is a rare opportunity for homeowners and investors alike. Move-in ready with room to grow!

ENTITIES:
0: AMENITY -> updated kitchen


In [17]:
add_entity(current_index, "3-bedroom", "BEDROOMS")
add_entity(current_index, "2-bathroom", "BATHROOMS")

Added BEDROOMS: 3-bedroom
Added BATHROOMS: 2-bathroom


In [18]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 6 / 50


In [19]:
current_index = review_next()


Record Index: 6
Listing ID: 197

TEXT:
 Location, Location, Location! Just nine lots from the glistening Pacific Ocean, this stunning Pierpont Bay Beach home offers unobstructed ocean and island views in one of Ventura's most sought-after beach communities. Located on desirable Marthas Vineyard Court, these rare street-to-street view homes are seldom available, making this an exceptional opportunity to own in this coveted coastal neighborhood. The versatile floor plan features a beautifully remodeled kitchen on the first floor, a dining room with French doors opening to a creatively designed, spacious patio ideal for entertaining, a remodeled bathroom, and three bedrooms--one currently utilized as a media/guest room with an adjoining half bathroom. Upstairs, prepare to be captivated by some of the most spectacular ocean, beach, and mountain views Pierpont Bay has to offer. Enjoy breathtaking sunrises and sunsets, watch sailboats drift across the horizon, and take in the stunning islan

In [20]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 7 / 50


In [21]:
current_index = review_next()


Record Index: 7
Listing ID: 184

TEXT:
 Welcome to your NEWLY BUILT 2026 Luxurious Single-Story residence ideally located in the PRIME LOCATION of Temple City, known for its award-winning schools & exceptional convenience. The impressive curb appeal features vibrant landscaping, a brand-new long driveway, & a charming porch that creates a warm & welcoming first impression. Inside, the thoughtfully designed open & functional floor plan offers 5 bedrooms & 5 full bathrooms, including a detached studio ADU ideal for guests, extended family, or flexible living space. Majority of the house is Just Newly Rebuilt! High ceilings & designer finishes enhance the interior, complemented by numerous new upgrades including chevron flooring, contemporary brass lighting fixtures, custom cabinetry, dual-pane windows, LED recessed lighting, new doors, premium hardware, elegant details, & much more! The spacious living room with a stylish fireplace flows seamlessly into the dining room and gourmet kitch

In [22]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 8 / 50


In [23]:
current_index = review_next()


Record Index: 8
Listing ID: 9

TEXT:
 BEST VALUE IN THE NEIGHBORHOOD! Welcome home to this bright and open 3-bedroom, 2.5-bathroom home located on a corner lot in the gated Parklane community. Lovingly cared for by the same family for decades, this 1,376 square feet, two-story residence offers the perfect blend of comfort, light, and community living. From the moment you step inside, you're greeted by an abundance of natural light pouring through the home's many windows, one of the seller's favorite features. Soaring vaulted ceilings in the living room create a dramatic sense of space and openness, making the home feel larger than its footprint suggests. It's the kind of home that immediately feels welcoming, and it's easy to see why the family room, with its comfortable, lived-in warmth, became the favorite gathering spot. The family room opens to a spacious kitchen featuring plenty of counter space, cabinetry and enough space for a dining table and chairs. Both the laundry area and 

In [24]:
add_entity(current_index, "3-bedroom", "BEDROOMS")
add_entity(current_index, "2.5-bathroom", "BATHROOMS")

Added BEDROOMS: 3-bedroom
Added BATHROOMS: 2.5-bathroom


In [25]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 9 / 50


In [26]:
current_index = review_next()


Record Index: 9
Listing ID: 104

TEXT:
 Experience the elegance of this rare and extensively updated Tapestry model home, ideally situated in a coveted corner location that floods the residence with natural light. Offering an expansive 2,415 square feet of beautifully designed living space, this exceptional home features 4 bedrooms and 4 bathrooms, providing the perfect blend of comfort, style, and luxury. The welcoming entry sitting room showcases a gas fireplace, oversized windows, and soaring 15-foot ceilings, flowing seamlessly into a formal dining area with custom built-ins. New tile and wood flooring throughout the home create a fresh, modern aesthetic and elevate its sophisticated appeal. The beautifully updated gourmet kitchen features brand-new stainless steel appliances, modern finishes, a center island with breakfast bar seating, and ample cabinetry. The kitchen opens to a cozy family room with a private balcony, perfect for BBQs and outdoor entertaining. The second floor f

In [27]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 10 / 50


In [28]:
current_index = review_next()


Record Index: 10
Listing ID: 200

TEXT:
 Significant price reduction ! Seller is open to offers , owner financing and joint venture . This exceptional zoned industrial property in San Bernardino County is perfect for any investor. It features the main house and duplex on site, situated on an approximately 41580 square feet lot. The zoning regulations allow for various uses . Currently being used as a front residential home with back duplex and garage conversion. All tenants are on a month to month lease. Front home features 4Br. 2 Full baths. Family and living room. Kitchen has been updated. Laminate flooring throughout. Back duplex has had a few upgrades in the last two years. Garage has been converted to a studio apartment. Each unit has laundry hook ups. Front tenant manages and maintains the property and has an agreement with the owner . Total potential monthly gross income ranges from $6000-$7000 a month. All this information has been provided but not guaranteed. Buyer is advised

In [29]:
gold_records[current_index]["verified"] = True

In [30]:
current_index = review_next()


Record Index: 11
Listing ID: 216

TEXT:
 Welcome to Canyon Lake living! This beautifully situated 3-bedroom, 2-bathroom condominium offers a comfortable and low-maintenance lifestyle in one of Southern California's premier gated, resort-style communities. The inviting floor plan features a spacious living area filled with natural light, a well-appointed kitchen with ample cabinetry and counter space, generous-sized bedrooms, and a private balcony that's perfect for relaxing or enjoying your morning coffee. Whether you're looking for a full-time residence, weekend getaway, or investment opportunity, this home provides the perfect blend of comfort and convenience. Enjoy all that Canyon Lake has to offer, including private lake access for boating, fishing, waterskiing, paddleboarding, and kayaking, along with an 18-hole golf course, community pools, parks, beaches, pickleball and tennis courts, equestrian center, clubhouse, waterfront dining, and year-round community events. Conveniently

In [31]:
add_entity(current_index, "3-bedroom", "BEDROOMS")
add_entity(current_index, "2-bathroom", "BATHROOMS")

Added BEDROOMS: 3-bedroom
Added BATHROOMS: 2-bathroom


In [32]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 12
Listing ID: 236

TEXT:
 This charming home located on a corner lot in the coveted S & S Eldorado Estates community is a masterful blend of Spanish-inspired elegance and modern refinement. Spanning 1,949 square feet of thoughtful living space on a 6,264-square-foot lot, this four-bedroom, three-bathroom residence has it all. The resort-style back yard includes a sparkling pool and spa that has just been remodeled with new resurfacing, tiles, coping and includes a new skimmer. The raised trex deck and covered patio with ceiling fan provides the perfect outdoor living space to enjoy while entertaining friends and family. Whether hosting under the stars or enjoying a quiet morning by the water, this outdoor haven is designed for indulgent relaxation. The interior of the home has been freshly-painted and the lower levels feature luxury vinyl plank flooring. Vaulted ceilings amplify the sense of airiness, while double-paned windows bathe each room in natural light, balancin

In [36]:
add_entity(current_index, "four-bedroom", "BEDROOMS")
add_entity(current_index, "three-bathroom", "BATHROOMS")

Added BEDROOMS: four-bedroom
Added BATHROOMS: three-bathroom


In [37]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 13
Listing ID: 240

TEXT:
 Welcome to this well maintained, single-story 2 bedroom, 1 bathroom cottage by the sea. Offering convenient Living in a practical and private layout. This home features air conditioning, vaulted ceilings, a built in storage closet on the patio, parking right in front, and a covered carport. Enjoy the enclosed patio, providing a private space for relaxing, entertaining, office, or a hobby space. A great opportunity for easy-care living in a friendly senior community that has many amenities such as golf, pickleball, swimming pool, hot tub, library, 6 clubhouses, and many activities. Enjoy a low-maintenance lifestyle in a quiet, friendly community close to local shopping, dining, parks and major commuter routes. Priced to sell!

ENTITIES:
0: BEDROOMS -> 2 bedroom
1: BATHROOMS -> 1 bathroom
2: AMENITY -> air conditioning
3: AMENITY -> patio
4: AMENITY -> parking
5: AMENITY -> carport
6: AMENITY -> patio
7: AMENITY -> swimming pool


In [38]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 14
Listing ID: 67

TEXT:
 Welcome to this beautifully maintained 2-bedroom, 2-bathroom condominium in the heart of Long Beach; located on one of the most charming tree-lined blocks. This coastal living home features an in-unit laundry, brand new dual pane windows, shutters and an HVAC mini split. Upon entering, you are invited to an abundance of natural light coming from your balcony, which presents a brand new sliding door and outdoor awning. Laminated wood flooring is offered throughout the living and dining rooms, kitchen and hallway. The sleek designed kitchen showcases stainless steel appliances. The primary bedroom provides its own en-suite bathroom while the additional room and bathroom add flexibility. Both bathrooms illustrate sophisticated black fixtures and quartz countertops. Stationed on the second floor, this property is complemented by refreshing ocean breezes. Complete with a detached 1 car garage and situated near The Pike, shopping, dining, entertainmen

In [39]:
add_entity(current_index, "2-bedroom", "BEDROOMS")
add_entity(current_index, "2-bathroom", "BATHROOMS")

Added BEDROOMS: 2-bedroom
Added BATHROOMS: 2-bathroom


In [40]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 15
Listing ID: 224

TEXT:
 Exceptional single-story residence offering striking curb appeal, leased solar, and beautiful city views, set in a quiet street location. An inviting driveway with smooth access and ample space creates a welcoming arrival and convenient parking for residents and guests. Inside, enjoy a bright open-concept layout with high ceilings and abundant natural light, seamlessly connecting the living and dining areas. The flexible floor plan is ideal for family living, entertaining, and a refined work-from-home or business setting. The open-concept kitchen, designed for hosting, features an island and generous counter space, flowing effortlessly into the main living areas-perfect for gatherings and everyday living. Step outside to a seamless indoor-outdoor experience. A well-maintained garden with mature landscaping enhances the peaceful setting and lifestyle appeal. Conveniently located near shopping, medical services, and community amenities. The prope

In [41]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 16
Listing ID: 194

TEXT:
 Value Range Pricing $849,000-$899,000! Seller willing to entertain 2/1 Buy-Down! Now is the time! Welcome to a beautiful single-family attached home nestled within the charming Placentia Knolls community. Surrounded by mature landscaping, peaceful walkways, and a nearby association pool, this exceptional residence offers a perfect lifestyle blend of comfort, style, and convenience. Three standout features immediately set this home apart: a spacious floor plan offering over 1,725 square feet, an expansive outdoor backyard space, and a low HOA of just $280 per month-and that's only the beginning! Step inside and you'll be welcomed by a neutral color palette, an abundance of natural light, soaring vaulted ceilings, and a spacious living room centered around an impressive fireplace, creating the perfect setting for relaxing or entertaining. The living area flows seamlessly into a generous dining space, ideal for everyday living and special gatherin

In [42]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 17
Listing ID: 15

TEXT:
 Family Living at Its Finest, across from Wilderness Park. Welcome to your forever home in one of South Redondo's most coveted neighborhoods. This beautifully remodeled 4-bedroom, 2-bathroom single family home offers the perfect blend of move-in-ready comfort and an unbeatable lifestyle location. Step through the street-level entry into a bright, open great room - living, dining, and kitchen flow seamlessly together, anchored by a large deck off the kitchen that's made for morning coffee or evening sunsets. The main level also features a private bedroom and full bathroom, ideal for guests, a home office, or single-level living. Downstairs, three additional bedrooms and a second full bathroom open directly to the grassy backyard a true kid-friendly retreat for play, pets, and weekend barbecues. Location, Location, Location. Set directly across from Wilderness Park, this home puts nature right outside your door. You're just a short walk or quick dr

In [43]:
add_entity(current_index, "4-bedroom", "BEDROOMS")
add_entity(current_index, "2-bathroom", "BATHROOMS")

Added BEDROOMS: 4-bedroom
Added BATHROOMS: 2-bathroom


In [ ]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 18
Listing ID: 177

TEXT:
 Welcome to 6855 Gunn Drive, a charming Mid Century Modern home tucked away in Oakland's Montclair hills. This 3 bedroom, 2 bathroom home offers approximately 1,675 square feet of living space on an 8,720 square foot lot. The bright, updated kitchen features white cabinetry, generous counter space and a large island that opens beautifully to the living areas. Expansive windows fill the home with natural light and showcase the surrounding greenery, while the living room fireplace adds warmth and character. Enjoy the enclosed sunroom, perfect for relaxing or entertaining, along with a wraparound deck surrounded by mature trees for a peaceful indoor outdoor living experience. An attached garage and desirable Montclair and Shepherd Canyon location complete this inviting hillside retreat.

ENTITIES:
0: BEDROOMS -> 3 bedroom
1: BATHROOMS -> 2 bathroom
2: SQFT -> 1,675 square feet
3: AMENITY -> updated kitchen
4: AMENITY -> fireplace
5: AMENITY -> deck

In [46]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 19
Listing ID: 24

TEXT:
 Thoughtfully updated and ideally situated within one of Aliso Viejo's most desirable communities, this beautifully maintained single-level home offers an elevated take on comfortable Southern California living. Featuring the same highly sought-after floor plan as neighboring homes, this residence stands apart with tasteful upgrades that enhance both style and functionality. The heart of the home is the updated kitchen, where modern finishes, refreshed cabinetry, and upgraded surfaces create a space that feels both current and inviting. The open-concept design allows the kitchen, dining area, and living room to flow together effortlessly, making entertaining easy and everyday living enjoyable. Updated windows further enhance the home's appeal, bringing in abundant natural light while improving comfort and energy efficiency. Step outside to a spacious private patio designed for relaxation and outdoor enjoyment. Whether hosting weekend gatherings, 

In [47]:
gold_records[current_index]["verified"] = True
current_index = review_next()


Record Index: 20
Listing ID: 247

TEXT:
 Turnkey! Cul-de-sac! University Grove! Beautifully maintained 5-bedroom, 3-bathroom home offering 3086 square feet. on a 7274 square feet. lot in the highly desirable University Grove community. Built in 2002 with a low $90/month HOA and located in the acclaimed Redlands Unified School District. Fresh interior paint and newer carpet make this home move-in ready. Flexible floor plan features a downstairs bedroom and full bathroom, ideal for guests, multigenerational living, or a home office. Bright, open living spaces flow into a spacious kitchen with center island, built-in wine fridge, walk-in pantry, and abundant cabinetry. The living room includes a cozy fireplace. Upstairs offers a spacious primary suite with sitting area potential, dual vanities, soaking tub, separate shower, and walk-in closet, plus three additional bedrooms, a built-in office/flex room, and a large game room. Enjoy a tropical-inspired backyard with covered patio, mature 

In [48]:
add_entity(current_index, "5-bedroom", "BEDROOMS")
add_entity(current_index, "3-bathroom", "BATHROOMS")

Added BEDROOMS: 5-bedroom
Added BATHROOMS: 3-bathroom


In [49]:
gold_records[current_index]["verified"] = True

In [50]:
gold_records[current_index]["verified"] = True
save_gold()

Verified: 21 / 50


In [51]:
import json

# Save your current corrections
save_gold()

# Keep only manually verified listings
verified_records = [
    record for record in gold_records
    if record["verified"] is True
]

print(f"Total verified listings: {len(verified_records)}")

# Create a smaller gold test file
output_path = "../data/week3_gold_test_21.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for record in verified_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved:", output_path)

Verified: 21 / 50
Total verified listings: 21
Saved: ../data/week3_gold_test_21.jsonl
